# 3. Deep Agents + LangChain middleware

This notebook moves from **capability** to **control**.

We demonstrate:

1. **Personally Identifiable Information (PII) middleware** — protect sensitive data.
2. **Human-in-the-loop middleware** — require approval before selected actions.
3. **Tool-call limits** — constrain cost and runaway behaviour.

Middleware is like checks at the doors around the agent rather than new intelligence inside the model.

In [1]:
from pathlib import Path

from dotenv import load_dotenv
from deepagents import create_deep_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    PIIMiddleware,
    ToolCallLimitMiddleware,
)
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()

MODEL = "gpt-5.6-luna"
model = ChatOpenAI(model=MODEL, use_responses_api=True)

## 1. PII middleware

PII means **Personally Identifiable Information**.

We start with email redaction and credit-card masking. The model itself is unchanged; middleware changes what reaches it.

In [2]:
pii_agent = create_deep_agent(
    model=model,
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
    ],
    system_prompt=(
        "Repeat the information you receive and explain which parts appear sensitive."
    ),
)

In [3]:
result = pii_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "The claimant email is sarah.jones@example.com and the payment card is "
            "4111 1111 1111 1111. The claim amount is £7,300."
        ),
    }]
})

print(result["messages"][-1].text)

Information received:
- Claimant email: `[REDACTED_EMAIL]`
- Payment card: ending in **1111**
- Claim amount: **£7,300**

Sensitive information:
- The email address is personal contact information, even though it is redacted.
- The card’s last four digits are payment-card information and can help identify an account when combined with other details.
- The claim amount is sensitive financial and potentially legal/insurance information.


### Add an insurer-specific identifier

Built-in detectors cannot know every organisation-specific identifier. Here we add a simple policy-number pattern.

In [4]:
policy_number_pattern = r"\bPOL-\d{6}\b"

policy_agent = create_deep_agent(
    model=model,
    middleware=[
        PIIMiddleware(
            "policy_number",
            detector=policy_number_pattern,
            strategy="redact",
            apply_to_input=True,
        )
    ],
)

result = policy_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Please summarise the claim on policy POL-123456.",
    }]
})

print(result["messages"][-1].text)

I don’t have access to any claim records or documents for policy **[REDACTED_POLICY_NUMBER]**. Please provide the claim correspondence, report, or policy details, and I can summarise it.


## 2. Human-in-the-loop middleware

The agent may propose a file write, but the write does not happen until a human approves it.

In [5]:
workspace = Path("middleware_workspace")
workspace.mkdir(exist_ok=True)


@tool
def read_demo_file(filename: str) -> str:
    """Read a text file from the teaching workspace."""
    return (workspace / filename).read_text(encoding="utf-8")


@tool
def write_demo_file(filename: str, content: str) -> str:
    """Write a text file to the teaching workspace."""
    (workspace / filename).write_text(content, encoding="utf-8")
    return f"Wrote {filename}"

In [6]:
approval_agent = create_deep_agent(
    model=model,
    tools=[read_demo_file, write_demo_file],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_demo_file": False,
                "write_demo_file": {
                    "allowed_decisions": ["approve", "reject"],
                },
            }
        )
    ],
    system_prompt=(
        "Help with simple file tasks. If a requested write is allowed by the tools, "
        "propose it normally and let the approval system handle the decision."
    ),
)

config = {"configurable": {"thread_id": "classroom-demo"}}

The next call should pause before writing the file.

In [7]:
paused = approval_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Write a file called note.txt containing: "
                "'Human approval is useful for actions with side effects.'"
            ),
        }]
    },
    config=config,
    version="v2",
)

print(paused.interrupts)

(Interrupt(value={'action_requests': [{'name': 'write_demo_file', 'args': {'filename': 'note.txt', 'content': 'Human approval is useful for actions with side effects.'}, 'description': "Tool execution requires approval\n\nTool: write_demo_file\nArgs: {'filename': 'note.txt', 'content': 'Human approval is useful for actions with side effects.'}"}], 'review_configs': [{'action_name': 'write_demo_file', 'allowed_decisions': ['approve', 'reject']}]}, id='d2f8925c3c62075fa3b773e7b80a5ea6'),)


Approve the proposed action using the **same thread ID**.

In [8]:
approved = approval_agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config=config,
    version="v2",
)

print(approved.value["messages"][-1].text)

Created `note.txt` containing:

`Human approval is useful for actions with side effects.`


In [9]:
print((workspace / "note.txt").read_text(encoding="utf-8"))

Human approval is useful for actions with side effects.


## 3. Tool-call limits

To make the classroom demonstration predictable, we use a small deterministic lookup tool rather than hoping a real web task makes too many searches.

In [10]:
@tool
def lookup_claim_fact(topic: str) -> str:
    """Return one short teaching fact about insurance claims."""
    facts = {
        "fraud": "Fraud referrals can create false positives.",
        "privacy": "Claims data may contain sensitive personal information.",
        "cost": "Repair costs can contribute to motor claims inflation.",
        "ai": "AI can help summarise claim notes but should not automatically replace judgement.",
    }
    return facts.get(topic.lower(), "No fact found.")

In [11]:
limited_agent = create_deep_agent(
    model=model,
    tools=[lookup_claim_fact],
    middleware=[
        ToolCallLimitMiddleware(
            tool_name="lookup_claim_fact",
            run_limit=2,
            exit_behavior="continue",
        )
    ],
    system_prompt=(
        "When the user asks for several claim facts, call lookup_claim_fact separately "
        "for each requested topic. Explain any tool-limit message you receive."
    ),
)

In [12]:
result = limited_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Look up these four topics separately: fraud, privacy, cost and AI. "
            "Then summarise all four."
        ),
    }]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Look up these four topics separately: fraud, privacy, cost and AI. Then summarise all four.
================================== Ai Message ==================================

[{'id': 'rs_095cc31ffedf0c94006a943334e49887d29ab8d5af8a603377', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqlDM1owSSZo4NdHNu-feSYUETz_CnhnEtWk_NG8jnGhQfETRsjU98gskqDn3-g6TQiCH5tug7mF8dvM4Cq5aSBy7A2RTm5-bLE6KBSpgtBlsTjbem4WDlKS_P88GMEqnX686-kzCElWm8umr6VOKiMwiBpcZM2yJrh4XbqPR817SYLOSefpbXBHHMr-z2CaPgJDcxRzpP9qZT0Fsa13WkpwxCnrouoo7oBNdeY8MQ3k_GmQQm50WggkNoMd_9XEy6zzo1HfLTXIoGbVhkGPg_4KitGMB0S-Io7C6DRdxkES-mTgNIYulXrfuV_vRRmHM_42Oo9Fk0NWLX9iHORCF2KwODo9q9dzn9pfz5A3YB_fqncwgfTdPIoMAlPpSAeKeZmg7O0Cvy1Y8AyHvCLxJxc9WH-EhNkhqQQf0oinTGYDg8JkwBBXTQ9eqNTL4b0_FDOO_JNa3FHGU_TdZZeenioEBGyynpTOgawQ-XlnxnHZunjq-KZmzEQTZ_qn8pP-ynaqoEiZMpr40iyRPB8o_wALV3ZdFYZJZJg-kfx7yXvZF1gX6Mb0me__4ZUyToT_jeqMxrlznB3sA1Vvj08iBzGq

## 4. Combine middleware

Middleware is composable. This agent protects email addresses and limits tool use.

In [13]:
combined_agent = create_deep_agent(
    model=model,
    tools=[lookup_claim_fact],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        ToolCallLimitMiddleware(
            tool_name="lookup_claim_fact",
            run_limit=2,
        ),
    ],
    system_prompt=(
        "Answer insurance questions using the teaching lookup tool when useful. "
        "Never invent the user's redacted personal information."
    ),
)

result = combined_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "My email is student.actuary@example.com. "
            "Look up the privacy and AI claim facts and explain why they matter."
        ),
    }]
})

print(result["messages"][-1].text)

I couldn’t retrieve specific teaching facts for either **privacy** or **AI in insurance claims**.

In general, they matter because:

- **Privacy:** Insurance claims often involve sensitive information, such as medical records, financial details, photos, and identifying data. Proper privacy protections help prevent unauthorized access, misuse, or disclosure.
- **AI:** Insurers may use AI to help process documents, estimate damage, detect fraud, or prioritize claims. Because AI can make errors or reflect biased data, claimants should be able to understand decisions, correct inaccurate information, and request human review where appropriate.

I also won’t use or expose the redacted email address you provided.


## What the three controls protect

| Middleware | Main thing protected |
|---|---|
| PII middleware | **Data** |
| Human-in-the-loop | **Actions** |
| Tool-call limits | **Resources, cost and runaway behaviour** |

### Common gotchas

- PII detection is not a complete data-governance programme.
- Built-in PII categories will not know every organisation-specific identifier.
- Human-in-the-loop needs state persistence and a stable thread ID.
- A tool-call limit does not prove that the remaining calls are sensible.
- Middleware order can matter in more advanced combinations.

### Related concepts

Model-call limits, retries, fallbacks, summarisation middleware, custom middleware hooks,
LangGraph deterministic workflows, LangSmith tracing and evaluation, Model Context Protocol,
and sandboxed execution backends.

## 5. Middleware around a real tool

The examples above used a small deterministic lookup tool so the demo stays predictable. Here we wrap `get_price_series_yahoo` from `middleware_workspace/prices.py` -- a real function that calls Yahoo Finance -- and put a tool-call limit around it, the same as we did for `lookup_claim_fact`.

In [ ]:
from middleware_workspace.prices import get_price_series_yahoo


@tool
def yahoo_price_series(ticker: str, period: str = "1y") -> str:
    """Look up a daily OHLCV price history for a stock ticker from Yahoo Finance.

    `period` is one of "1mo", "3mo", "6mo", "1y", "2y", "5y", "10y", "ytd", "max".
    Returns the oldest and most recent rows plus the row count, not the full series,
    to keep the tool's reply short enough for the agent to read easily.
    """
    rows = get_price_series_yahoo(ticker, period=period)
    first, last = rows[0], rows[-1]
    return (
        f"{ticker.upper()}: {len(rows)} trading days from {first['date']} to {last['date']}.\n"
        f"First close: {first['close']}\n"
        f"Last close: {last['close']}"
    )


price_agent = create_deep_agent(
    model=model,
    tools=[yahoo_price_series],
    middleware=[
        ToolCallLimitMiddleware(
            tool_name="yahoo_price_series",
            run_limit=2,
        )
    ],
    system_prompt=(
        "Use yahoo_price_series to answer share-price questions. "
        "Explain any tool-limit message you receive."
    ),
)

result = price_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "How have LGEN.L and AIICO.LG done over the last year?",
    }]
})

print(result["messages"][-1].text)